In [1]:
import numpy as np
import os as os
import pickle as pickle
import numpy as np
import os as os
import tensorflow as tf
tfk = tf.keras

/Users/hamzaahmed/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from cymetric.pointgen.pointgen import PointGenerator

In [3]:
# ---------------------------------------------------------------------------
#  RP³ special-Lagrangian finder in the Fermat quintic  ⟨ ℙ⁴ | Σ z_i⁵ = 0 ⟩
#
#  – bulk points   : Fermat-quintic cloud via cymetric.pointgen.PointGenerator
#  – RP³ “signal”  : analytic real-locus sampler with low noise
#  – pipeline      :
#        1) residual pre-filter
#        2) local-PCA rank-3 filter
#        3) [coords‖projector‖residual] feature
#        4) adaptive-ε DBSCAN (min_samples=5)
#        5) sweep residual to recover all inliers
#        6) print the learned RP³ defining equations deviations
# ---------------------------------------------------------------------------

import numpy as np
from sklearn.neighbors     import NearestNeighbors
from sklearn.cluster       import DBSCAN
from sklearn.preprocessing import StandardScaler
from cymetric.pointgen.pointgen import PointGenerator

np.random.seed(7)

# ----------------------------------------------------------------------
# A)  Point generators
# ----------------------------------------------------------------------
monomials    = 5 * np.eye(5, dtype=np.int64)
coefficients = np.ones(5)
kmoduli      = np.ones(1)
ambient      = np.array([4])

pg = PointGenerator(monomials, coefficients, kmoduli, ambient)

def quintic_bulk(n=8_000):
    """Return (n,10) real array of random points on the Fermat quintic (z0=1 patch)."""
    Z = pg.generate_points(n)  # (n,5) complex
    return np.column_stack([Z.real, Z.imag])

def rp3_cloud(n=1_000, noise=1e-3):
    """
    Return (n,10) real array of noisy RP³ samples on the quintic,
    using two Newton steps for accurate projection.
    """
    rng = np.random.default_rng()
    pts = []
    for _ in range(n):
        x123 = rng.uniform(-1, 1, 3)
        S    = 1.0 + np.sum(x123**5)
        x4   = -np.sign(S) * np.abs(S)**0.2  # solve x4^5 = -S
        z    = np.array(x123.tolist() + [x4], complex)
        z   += noise * (rng.standard_normal(4) + 1j*rng.standard_normal(4))
        for _ in range(2):                   # two Newton steps
            f, df = np.sum(z**5) + 1.0, 5*z**4
            lam   = f / np.sum(df * np.conj(df)).real
            z    -= lam * df
        z_full = np.hstack([[1+0j], z])      # include z0 = 1
        pts.append(np.column_stack([z_full.real, z_full.imag]).ravel())
    return np.asarray(pts, float)

# ----------------------------------------------------------------------
# B)  Assemble mixed cloud
# ----------------------------------------------------------------------
X_bulk, X_rp3 = quintic_bulk(), rp3_cloud()
X             = np.vstack([X_bulk, X_rp3])
print("cloud size:", X.shape)  # expect (9000, 10)

# ----------------------------------------------------------------------
# C)  RP³ residual & pre-filter
# ----------------------------------------------------------------------
def rp3_residual(P):
    """Compute max(max_i |Im z_i|, |Σ_i Re(z_i)^5|) for P shape (N,10)."""
    x = P[:, 0::2]    # real parts z0..z4
    y = P[:, 1::2]    # imag parts
    return np.maximum(np.max(np.abs(y), axis=1),
                      np.abs(np.sum(x**5, axis=1)))

res      = rp3_residual(X)
mask_res = res < 1e-2
X_filt   = X[mask_res]
idx_filt = np.where(mask_res)[0]
print("after residual filter:", X_filt.shape[0])  # ~1000

# ----------------------------------------------------------------------
# D)  Local PCA rank-3 test
# ----------------------------------------------------------------------
nbrs = NearestNeighbors(n_neighbors=11).fit(X_filt)
_, nn = nbrs.kneighbors(X_filt)

eig_ratio = np.empty(len(X_filt))
for i in range(len(X_filt)):
    C   = np.cov(X_filt[nn[i,1:]], rowvar=False)
    lam = np.sort(np.linalg.eigvalsh(C))[::-1]
    eig_ratio[i] = lam[3] / lam[0]

lam_thresh = 1e-2      # loosened threshold
cand_mask  = eig_ratio < lam_thresh
cand_idx   = np.where(cand_mask)[0]
print("rank-3 candidates:", len(cand_idx))  # ~950

# ----------------------------------------------------------------------
# E)  Build [coords | projector | residual] features + z-score
# ----------------------------------------------------------------------
proj = []
for i in cand_idx:
    C       = np.cov(X_filt[nn[i,1:]], rowvar=False)
    _, vecs = np.linalg.eigh(C)
    V3      = vecs[:, -3:]
    proj.append((V3 @ V3.T).ravel())

feat = np.hstack([
    X_filt[cand_idx],                      # 10 dims
    np.vstack(proj),                       # 100 dims
    res[mask_res][cand_idx].reshape(-1,1)  # 1 dim
])
feat = StandardScaler().fit_transform(feat)

# ----------------------------------------------------------------------
# F)  Adaptive-ε DBSCAN (min_samples=5)
# ----------------------------------------------------------------------
nbr_feat = NearestNeighbors(n_neighbors=6).fit(feat)
dists, _ = nbr_feat.kneighbors(feat)
eps0     = np.percentile(dists[:,5], 90)
eps      = eps0
clusters = []

while eps <= 5 * eps0 and not clusters:
    labels   = DBSCAN(eps=eps, min_samples=5).fit_predict(feat)
    clusters = [cand_idx[labels==l] for l in set(labels) if l != -1]
    eps     *= 1.4

if not clusters:
    raise RuntimeError("DBSCAN failed – adjust thresholds")

clusters_orig = [idx_filt[c] for c in clusters]
largest       = max(clusters_orig, key=len)
print(f"DBSCAN → ε≈{eps/1.4:.3f}, cluster size={len(largest)}")

# ----------------------------------------------------------------------
# G)  Final sweep: recover all RP³ inliers by residual
# ----------------------------------------------------------------------
inliers = np.where(res < 1e-2)[0]
print("final RP³ inliers (res<1e-2):", len(inliers))  # ≈1000

# ----------------------------------------------------------------------
# H)  Print learned RP³ defining equations deviations
# ----------------------------------------------------------------------
cluster_pts = X[largest]      # (cluster_size, 10)
x = cluster_pts[:, 0::2]      # real parts z0..z4
y = cluster_pts[:, 1::2]      # imag parts

max_imag  = np.max(np.abs(y), axis=0)
sum_re5   = np.sum(x**5, axis=1)

print("\nLearned RP³ satisfies approximately:")
for i in range(5):
    print(f"  max |Im(z_{i})| = {max_imag[i]:.2e}")
print(f"  max |Σ_i=0..4 Re(z_i)^5| = {np.max(np.abs(sum_re5)):.2e}")


cloud size: (9005, 10)
after residual filter: 1000
rank-3 candidates: 837
DBSCAN → ε≈10.027, cluster size=796
final RP³ inliers (res<1e-2): 1000

Learned RP³ satisfies approximately:
  max |Im(z_0)| = 0.00e+00
  max |Im(z_1)| = 3.11e-03
  max |Im(z_2)| = 3.24e-03
  max |Im(z_3)| = 3.30e-03
  max |Im(z_4)| = 1.71e-03
  max |Σ_i=0..4 Re(z_i)^5| = 4.36e-05


In [4]:
# ----------------------------------------------------------------------
# L)  Persistent homology for b₂ (2-dimensional cycles)
# ----------------------------------------------------------------------
import numpy as np
from ripser import ripser

# cluster_pts should already be your learned RP³ point cloud:
# cluster_pts = X[largest]

# 1) compute Vietoris–Rips persistence up to H₂
dgms = ripser(cluster_pts, maxdim=2)['dgms']
H2   = dgms[2]    # array of shape (m,2): birth/death pairs for 2‐cycles

# 2) gap‐based Betti estimator (same as before)
def gap_betti(diag):
    if diag.size == 0:
        return 0
    lifespans = np.sort(diag[:,1] - diag[:,0])[::-1]
    if lifespans.size == 1:
        return 1
    gaps = lifespans[:-1] - lifespans[1:]
    return np.argmax(gaps) + 1

b2 = gap_betti(H2)
print(f"estimated second Betti number b₂ (over 𝔽₂) = {b2}")


estimated second Betti number b₂ (over 𝔽₂) = 1


In [7]:
# ----------------------------------------------------------------------
# J)  Robust estimation of b₁ via the largest‐gap threshold
# ----------------------------------------------------------------------
import numpy as np
from ripser import ripser

# use the *cluster* points, not the full inliers
cluster_pts = X[largest]    # shape (≈760,10)

# 1) compute H₁ with Ripser
dgms = ripser(cluster_pts, maxdim=1)['dgms']
H1   = dgms[1]              # array of shape (m,2): (birth, death)

# 2) compute lifespans and sort descending
lifespans = H1[:,1] - H1[:,0]
sorted_ls = np.sort(lifespans)[::-1]

# 3) find the largest gap between adjacent sorted lifespans  Cell In[18], line 1
gaps = sorted_ls[:-1] - sorted_ls[1:]
idx_largest_gap = np.argmax(gaps)

# 4) choose the threshold as the midpoint of that gap
thr = (sorted_ls[idx_largest_gap] + sorted_ls[idx_largest_gap+1]) / 2

# 5) count bars above threshold
b1 = np.sum(lifespans > thr)

# 6) report everything for diagnostics
print("Top 10 H₁ lifespans:", np.round(sorted_ls[:10], 4))
print("Largest gap between lifespans at index", idx_largest_gap,
      f"({gaps[idx_largest_gap]:.4f})")
print(f"Threshold set to {(thr):.4f}")
print(f"estimated 1st Betti number b₁ = {b1}")


Top 10 H₁ lifespans: [0.206  0.1677 0.1591 0.154  0.152  0.1478 0.1412 0.1402 0.1395 0.1373]
Largest gap between lifespans at index 0 (0.0383)
Threshold set to 0.1869
estimated 1st Betti number b₁ = 1
